In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
# sys.path.append("/hdd/zijianwang/openpi/third_party/LIBERO-PRO")
sys.path.append("/home/zijianwang/openpi/third_party/LIBERO-PRO")
import collections
import dataclasses
import json
import logging
import math
import os
import cv2
import pathlib
import time
from datetime import datetime
from typing import Dict, Any
import imageio
import numpy as np
import perturbation
import tqdm
import yaml
from PIL import Image
import matplotlib.pyplot as plt

from libero.libero import benchmark
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from openpi_client import image_tools
from openpi_client import websocket_client_policy as _websocket_client_policy
from utils.interfaces import LMP_interface
from util import (
    compute_eef_trajectory_from_actions, 
    build_reusable_value_map, 
    evaluate_trajectory_with_value_map,
    is_gripper_closed,
    search_best_action_chunk,
    generate_candidate_action_chunks,
    select_best_action_chunk_using_valuemap,
    select_best_action_chunk_using_angles,
    detect_pre_grasp_state,
    get_reordered_objects,

)
print("✓ All libraries imported successfully")

LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]

11111111111111111111111


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/zijianwang/openpi/examples/libero/.venv/lib/python3.8/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
✓ All libraries imported successfully


In [2]:
# Constants
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]
LIBERO_ENV_RESOLUTION = 256  # resolution used to render training data

# Helper functions
def _quat2axisangle(quat):
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

def _get_libero_env(task, resolution, seed):
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)
    return env, task_description

print("✓ Constants and helper functions defined")


def setup_logging(args):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = f"LIBERO-PRO-{args['task_suite_name']}-{timestamp}"
    if args.get('run_id_note') is not None:
        run_id += f"-{args['run_id_note']}"

    os.makedirs(args['local_log_dir'], exist_ok=True)
    log_filepath = os.path.join(args['local_log_dir'], f"{run_id}.txt")

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[logging.StreamHandler(), logging.FileHandler(log_filepath, encoding="utf-8")],
    )

    logger = logging.getLogger(__name__)
    logger.info(f"Experiment run ID: {run_id}")
    logger.info(f"Log file path: {log_filepath}")

    return logger, run_id, log_filepath


def save_experiment_config(args, run_id, log_filepath):
    if not args.get('save_experiment_config', True):
        return

    config_data = {
        "run_id": run_id,
        "timestamp": datetime.now().isoformat(),
        "args": args,
        "log_file": log_filepath,
    }

    config_filepath = os.path.join(args['local_log_dir'], f"{run_id}_config.json")
    with open(config_filepath, "w", encoding="utf-8") as f:
        json.dump(config_data, f, indent=2, ensure_ascii=False)

    logging.info(f"Experiment configuration saved to: {config_filepath}")


def save_episode_video(replay_images, task_description, episode_idx, success, run_id, args):
    task_segment = task_description.replace(" ", "_").replace("/", "_")
    task_video_dir = os.path.join(args['video_out_path'], run_id, task_segment)
    os.makedirs(task_video_dir, exist_ok=True)

    suffix = "success" if success else "failure"
    video_filename = f"episode_{episode_idx:03d}_{suffix}.mp4"
    video_filepath = os.path.join(task_video_dir, video_filename)

    try:
        imageio.mimwrite(
            video_filepath,
            [np.asarray(x) for x in replay_images],
            fps=10,
        )
        logging.info(f"Episode video saved: {video_filepath}")
    except Exception as e:
        logging.error(f"Failed to save episode video: {e}")

  
def _plot_trajectory(trajectory, output_path):
    """
    Plot trajectory with N dimensions as different colored lines.
    
    Args:
        trajectory: List of N-dimensional velocity vectors or 1D array
        output_path: Path to save the plot
    """
    if len(trajectory) == 0:
        logging.warning("No velocity data to plot")
        return
    
    velocity_array = np.array(trajectory)  # Shape: (T, N) or (T,)
    time_steps = np.arange(len(trajectory))
    
    # Handle 1D case (T,) by reshaping to (T, 1)
    if velocity_array.ndim == 1:
        velocity_array = velocity_array.reshape(-1, 1)
    
    # Get number of dimensions
    num_dims = velocity_array.shape[1]
    
    # Create figure with good size
    plt.figure(figsize=(12, 6))
    
    # Define colors - use a colormap for arbitrary number of dimensions
    colors = plt.cm.tab10(np.linspace(0, 1, max(num_dims, 1)))
    dimension_labels = [f'Joint {i+1}' for i in range(num_dims)]
    
    # Plot each dimension with different color
    for dim in range(num_dims):
        plt.plot(time_steps, velocity_array[:, dim], 
                color=colors[dim], label=dimension_labels[dim], linewidth=1.5, alpha=0.8)
    
    plt.xlabel('Time Step', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.title('Trajectory', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save the plot
    plt.savefig(output_path, dpi=100)
    plt.close()
    
    logging.info(f"Plot saved to: {output_path}")


def add_text_to_image(temp_img, text):
    """Add text overlay to image showing length and step number.
    
    Args:
        temp_img (np.ndarray): Input image of shape (224, 224, 3)
        text (str): Text to display, can contain newlines
        
    Returns:
        np.ndarray: Image with text overlay
    """
    img = temp_img.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.4
    thickness = 1
    
    # Split text by newlines
    lines = text.split('\n')
    
    # Calculate total text dimensions
    line_heights = []
    max_width = 0
    
    for line in lines:
        (line_width, line_height), _ = cv2.getTextSize(line, font, font_scale, thickness)
        line_heights.append(line_height)
        max_width = max(max_width, line_width)
    
    # Calculate line spacing (add some padding between lines)
    line_spacing = max(line_heights) + 5 if line_heights else 0
    total_height = len(lines) * line_spacing
    
    # Position text 10 pixels from right and top edges
    text_x = img.shape[1] - max_width - 10
    start_y = max(line_heights) + 10 if line_heights else 10
    
    # Draw each line
    for i, line in enumerate(lines):
        y_position = start_y + i * line_spacing
        
        # Add white text with black outline for visibility
        cv2.putText(img, line, (text_x, y_position), font, font_scale, (0,0,0), 2)
        cv2.putText(img, line, (text_x, y_position), font, font_scale, (255,255,255), 1)
    
    return img

print("✓ Logging and config management functions defined")


def sample_positions(env_vox, task_description, start_obs, target_objects, avoid_objects):
    lmp_env = LMP_interface(env_vox, task_description)
    movable_gripper, gripper_map, affordance_map, avoidance_map = lmp_env.build_value_map(target_objects, avoid_objects)
    """
    Type annotation for movable_gripper:
    movable_gripper: dict
        A dictionary containing information about the gripper, with the following keys:
        - name (str): Name of the gripper, e.g., 'gripper'
        - position (np.ndarray): Array of shape (3,) with dtype int32, representing the voxel coordinates [x, y, z]
        - aabb (np.ndarray): Array of shape (2, 3) with dtype int32, representing the axis-aligned bounding box
        - _position_world (np.ndarray): Array of shape (3,) representing the world coordinates [x, y, z]
    """
    info = lmp_env.sample_pos(movable_gripper, affordance_map=affordance_map, avoidance_map=avoidance_map, gripper_map=gripper_map)
    
    # obs['robot0_gripper_qpos']
    joint_pos = {'robot0_joint_pos': start_obs['robot0_joint_pos'], 'robot0_gripper_qpos': start_obs['robot0_gripper_qpos']}
    joint_pos_sample = [joint_pos]
    
    joint_pos_sample.extend(info['sample_joint_pos'])

    sampled_action = info['sample_action']
    
    return joint_pos_sample, sampled_action, info['target_xyz']
    
def sample_trajctories(env, sim_state, joint_pos_sample, cfg, model, task_description, processor, action_head, proprio_projector, noisy_action_projector):
    
    pos_info = {}
    traj = []
    start_pos = []
    start_joint_pos = []
    num_open_loop_steps = 8
    resize_size = 224
    model_family = "openvla"
    # action_queue = deque(maxlen=num_open_loop_steps)
    # object_names = sim_clone.get_object_names()
    for i in range(len(joint_pos_sample)):
        joint_pos = joint_pos_sample[i]
        
        env.reset(sim_state)
        env.restore_robot_joint_positions(joint_pos['robot0_joint_pos'], joint_pos['robot0_gripper_qpos'])
        # print(joint_pos)
        obs, reward, done, info = env.step(LIBERO_DUMMY_ACTION)
        start_pos.append(obs["robot0_eef_pos"])
        start_joint_pos.append({'robot0_joint_pos': obs['robot0_joint_pos'], 'robot0_gripper_qpos': obs['robot0_gripper_qpos']})
        observation, img = prepare_observation(obs, resize_size)
        # Image.fromarray(np.uint8(img)).save("./tmp/output_image_2.png")

        actions = get_action(
            cfg,
            model,
            observation,
            task_description,
            processor=processor,
            action_head=action_head,
            proprio_projector=proprio_projector,
            noisy_action_projector=noisy_action_projector,
            use_film=cfg.use_film,
        )
        
        traj_pos = []
        replay_imgs = []
        # 在克隆环境中执行动作，记录轨迹
        for i in range(num_open_loop_steps):
        
            action = process_action(actions[i], model_family)
            obs, reward, done, info  = env.step(action.tolist())
            ee_pos = obs["robot0_eef_pos"]
            img = obs["agentview_image"]
            traj_pos.append(ee_pos)
            replay_imgs.append(img)
        traj.append(np.array(traj_pos))
        
    

    pos_info['traj_world'] = traj
    pos_info['sample_pos_world'] = start_pos
    pos_info['start_joint_pos'] = start_joint_pos
    
    
    return pos_info

def calculate_similarity_between_actions(cost_sampled_action, candidate_action_chunks):
    """
    Calculate the cosine similarity between cost_sampled_action and each candidate action chunk.
    
    Args:
        cost_sampled_action: A single action (numpy array or list)
        candidate_action_chunks: List of candidate actions (numpy arrays or lists)
    
    Returns:
        List of cosine similarities between cost_sampled_action and each candidate
    """
    
    # Convert to numpy arrays if not already
    sampled_action = np.array(cost_sampled_action)[0][:3]
    similarities = []
    
    for candidate_chunk in candidate_action_chunks:
        candidate = np.array(candidate_chunk)[0][:3]
        
        # Ensure both vectors have the same dimension
        if sampled_action.shape != candidate.shape:
            raise ValueError(f"Dimension mismatch: sampled_action {sampled_action.shape}, candidate {candidate.shape}")
        
        # Normalize vectors
        sampled_norm = np.linalg.norm(sampled_action)
        candidate_norm = np.linalg.norm(candidate)
        
        if sampled_norm == 0 or candidate_norm == 0:
            # If either vector is zero, similarity is undefined, return 0
            similarities.append(0.0)
        else:
            # Calculate cosine similarity using dot product
            cos_sim = np.dot(sampled_action, candidate) / (sampled_norm * candidate_norm)
            # Clamp cosine value to [-1, 1] to avoid numerical issues
            cos_sim = np.clip(cos_sim, -1.0, 1.0)
            similarities.append(cos_sim)
    
    return similarities

def generate_action_sequence_fixed_steps(current_xyz, target_xyz, num_steps, normalization_factor_pos=0.1, gripper_action=-1):
    """
    生成一个固定步数 (num_steps) 的动作序列，用于将机械臂末端从当前位置移动到目标位置。
    这个函数假设直线路径且无障碍。

    参数:
    current_xyz (list/np.array): 起始 [x, y, z] 坐标。
    target_xyz (list/np.array): 目标 [x, y, z] 坐标。
    num_steps (int): 期望的总步数。
    normalization_factor_pos (float): LIBERO 环境的最大动作增量值 (通常是 0.1)。
    gripper_action (int/float): 夹持器动作值 (例如 -1 表示打开, 1 表示关闭)。

    返回:
    list: 一个包含 num_steps 个 7 维动作数组 (action chunks) 的序列。
    """
    current_xyz = np.array(current_xyz)
    target_xyz = np.array(target_xyz)
    
    if num_steps <= 0:
        return []

    # 计算每一步的精确增量位移
    delta_pos = (target_xyz - current_xyz) / num_steps
    
    # 将增量归一化到 LIBERO 的 [-1, 1] 范围内
    normalized_delta_pos = delta_pos / normalization_factor_pos
    normalized_delta_pos = np.clip(normalized_delta_pos, -1.0, 1.0)

    # 姿态 (Orientation) 增量设为 0（保持当前姿态）
    delta_rot = np.zeros(3) 
    
    # 构造单个动作原型
    action_chunk_prototype = np.concatenate([normalized_delta_pos, delta_rot, [gripper_action]])

    # 生成完整的动作序列
    action_sequence = [action_chunk_prototype] * num_steps
    
    return action_sequence

# # 示例用法
# start_point = [0.1, 0.2, 0.3]
# end_point = [0.5, 0.5, 0.6]

# # 示例 1: 使用 5 步到达目标
# steps_5 = generate_action_sequence_fixed_steps(start_point, end_point, num_steps=5)
# print(f"生成的 5 步动作序列长度: {len(steps_5)}")
# print(f"每一步的动作值 (5步): {steps_5[0]}")


# # 示例 2: 使用 10 步到达目标
# steps_10 = generate_action_sequence_fixed_steps(start_point, end_point, num_steps=10)
# print(f"生成的 10 步动作序列长度: {len(steps_10)}")
# print(f"每一步的动作值 (10步): {steps_10[0]}")


# Example usage:
# similarities = calculate_similarity_between_actions(cost_sampled_action, candidate_action_chunks)


✓ Constants and helper functions defined
✓ Logging and config management functions defined


In [3]:
# ============================================================================
# Model server parameters
# ============================================================================
host = "0.0.0.0"
port = 8001
resize_size = 224
replan_steps = 5
sampling_bs = 5
sampling_std = 4
search_start_step = 0  # Only use search_best_action_chunk when t >= this value
# ============================================================================
# LIBERO environment-specific parameters
# ============================================================================
task_suite_name = "libero_spatial_displacement"  # Options: libero_spatial, libero_object, libero_goal, libero_10, libero_90
num_steps_wait = 10  # Number of steps to wait for objects to stabilize in sim
num_trials_per_task = 50  # Number of rollouts per task
# ============================================================================
# LIBERO Pro parameters
# ============================================================================
evaluation_config_path = "/home/zijianwang/openpi/third_party/LIBERO-PRO/evaluation_config.yaml"
# ============================================================================
# Logging and experiment tracking parameters
# ============================================================================
local_log_dir = "./experiments/logs"
run_id_note = None
save_experiment_config_flag = True
# ============================================================================
# Video output
# ============================================================================
video_out_path = "./experiments/videos/"
img_out_path = "./experiments/imgs/"
traj_raw_data_path = "./experiments/traj_raw_data/"
# ============================================================================
# Random seed
# ============================================================================
seed = 7
# ============================================================================
# Combine all args into a dictionary
# ============================================================================
args = {
    'host': host,
    'port': port,
    'resize_size': resize_size,
    'replan_steps': replan_steps,
    'sampling_bs': sampling_bs,
    'task_suite_name': task_suite_name,
    'num_steps_wait': num_steps_wait,
    'num_trials_per_task': num_trials_per_task,
    'evaluation_config_path': evaluation_config_path,
    'local_log_dir': local_log_dir,
    'run_id_note': run_id_note,
    'save_experiment_config': save_experiment_config_flag,
    'video_out_path': video_out_path,
    'img_out_path': img_out_path,
    'traj_raw_data_path': traj_raw_data_path,
    'seed': seed,
    'sampling_std': sampling_std,
    'search_start_step': search_start_step,
}

print("✓ Hyperparameters configured:")
for key, value in args.items():
    print(f"  {key}: {value}")

✓ Hyperparameters configured:
  host: 0.0.0.0
  port: 8001
  resize_size: 224
  replan_steps: 5
  sampling_bs: 5
  task_suite_name: libero_spatial_displacement
  num_steps_wait: 10
  num_trials_per_task: 50
  evaluation_config_path: /home/zijianwang/openpi/third_party/LIBERO-PRO/evaluation_config.yaml
  local_log_dir: ./experiments/logs
  run_id_note: None
  save_experiment_config: True
  video_out_path: ./experiments/videos/
  img_out_path: ./experiments/imgs/
  traj_raw_data_path: ./experiments/traj_raw_data/
  seed: 7
  sampling_std: 4
  search_start_step: 0


In [5]:
# Setup logging
logging.basicConfig(level=logging.INFO)
logger, run_id, log_filepath = setup_logging(args)

# Save experiment configuration
save_experiment_config(args, run_id, log_filepath)

# Set random seed
np.random.seed(args['seed'])

# Initialize LIBERO task suite
benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[args['task_suite_name']]()
num_tasks_in_suite = task_suite.n_tasks
logging.info(f"Task suite: {args['task_suite_name']}")

pathlib.Path(args['video_out_path']).mkdir(parents=True, exist_ok=True)

# Determine max_steps based on task suite
if "libero_spatial" in args['task_suite_name']:
    max_steps = 300
elif "libero_object" in args['task_suite_name']:
    max_steps = 280
elif "libero_goal" in args['task_suite_name']:
    max_steps = 300
elif "libero_10" in args['task_suite_name']:
    max_steps = 520
elif "libero_90" in args['task_suite_name']:
    max_steps = 400
else:
    raise ValueError(f"Unknown task suite: {args['task_suite_name']}")

client = _websocket_client_policy.WebsocketClientPolicy(args['host'], args['port'])

# Start evaluation
total_episodes, total_successes = 0, 0

logger.info(f"Starting evaluation of task suite: {args['task_suite_name']}")
logger.info(f"Number of tasks: {num_tasks_in_suite}")
logger.info(f"Trials per task: {args['num_trials_per_task']}")

print("✓ Initialization complete, starting task loop...")

INFO:__main__:Experiment run ID: LIBERO-PRO-libero_spatial_displacement-20251107_142717
INFO:__main__:Log file path: ./experiments/logs/LIBERO-PRO-libero_spatial_displacement-20251107_142717.txt
INFO:root:Experiment configuration saved to: ./experiments/logs/LIBERO-PRO-libero_spatial_displacement-20251107_142717_config.json
INFO:root:Task suite: libero_spatial_displacement
INFO:root:Waiting for server at ws://0.0.0.0:8001...
INFO:__main__:Starting evaluation of task suite: libero_spatial_displacement
INFO:__main__:Number of tasks: 10
INFO:__main__:Trials per task: 50


[info] Using default task order for benchmark 'libero_spatial_displacement' (10 tasks).
✓ Initialization complete, starting task loop...


In [ ]:
non_valid_episodes = {1: [11, 13, 14, 15, 24, 28, 48]}

object_path = "/home/zijianwang/openpi/third_party/LIBERO-PRO/reordered_objects_with_descriptions.json"

# Get current timestamp for this run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_out_dir = video_out_dir = pathlib.Path(args['video_out_path']) / f"{timestamp}_{args['task_suite_name']}"
video_out_dir.mkdir(parents=True, exist_ok=True)
img_out_dir = pathlib.Path(args['img_out_path']) / f"{timestamp}_{args['task_suite_name']}"
img_out_dir.mkdir(parents=True, exist_ok=True)
traj_raw_data_dir = pathlib.Path(args['traj_raw_data_path']) / f"{timestamp}_{args['task_suite_name']}"

traj_raw_data_dir.mkdir(parents=True, exist_ok=True)
logging.info(f"Videos will be saved to: {video_out_dir}, Images will be saved to: {img_out_dir}, Trajectory raw data will be saved to: {traj_raw_data_dir}")
for task_id in tqdm.tqdm(range(num_tasks_in_suite)):
    if task_id != 1:
        continue
    
    # Get task
    task = task_suite.get_task(task_id)
    initial_states = task_suite.get_task_init_states(task_id)
    env, task_description = _get_libero_env(task, LIBERO_ENV_RESOLUTION, args['seed'])

    all_target_objects = get_reordered_objects(object_path, task_description)
    num_stages = len(all_target_objects)

    # Start episodes
    task_episodes, task_successes = 0, 0
    logger.info(f"\n***Starting task: {task_description}, This task has {num_stages} stages.")

    all_episode_costs = []
    current_episode_costs = []

    ### Entering task loop
    for episode_idx in tqdm.tqdm(range(args['num_trials_per_task'])):
        if episode_idx in non_valid_episodes[task_id]:
            continue

        use_search = False
        is_pre_grasp = False
        is_next_stage = False
        has_rebuilt = False
        current_stage = 0
        max_angle = None

        # if episode_idx != 0:
        #     continue
        
        logger.info(f"\nTask: {task_description}")
        logger.info(f"Episode {episode_idx + 1}/{args['num_trials_per_task']}")
        # Reset environment
        env.reset()
        robot_instance = env.robots[0]
        action_plan = collections.deque()
        obs = env.set_init_state(initial_states[episode_idx])
        ### Build reusable valuemap before task starts
        reusable_valuemap = build_reusable_value_map(env, task_description, stage=current_stage)
        env_vox = reusable_valuemap["env_vox"]
        # reusable_valuemap = None   

        # Setup
        t = 0
        replay_images, current_episode_costs, linear_speed_trajectory, gripper_state_trajectory = [], [], [], []
        logger.info(f"Starting episode {task_episodes + 1}...")
        while t < max_steps + args['num_steps_wait']:
            if is_next_stage == True and num_stages > 1 and has_rebuilt == False:
                current_stage += 1
                logging.info(f"开始构建新的可重复使用ValueMap: 目标对象: {reusable_valuemap['target_objects']}, 避免对象: {reusable_valuemap['avoid_objects']}")
                reusable_valuemap = build_reusable_value_map(env, task_description, stage=current_stage)
                has_rebuilt = True
            # IMPORTANT: Do nothing for the first few timesteps because the simulator drops objects
            if t < args['num_steps_wait']:
                obs, reward, done, info = env.step(LIBERO_DUMMY_ACTION)
                linear_speed, gripper_state = 0, 0
                linear_speed_trajectory.append(linear_speed)
                gripper_state_trajectory.append(gripper_state)
                t += 1
                continue

            # Get preprocessed image - note the 180 degree rotation
            img = np.ascontiguousarray(obs["agentview_image"][::-1, ::-1])
            wrist_img = np.ascontiguousarray(obs["robot0_eye_in_hand_image"][::-1, ::-1])
            img = image_tools.convert_to_uint8(image_tools.resize_with_pad(img, args['resize_size'], args['resize_size']))
            wrist_img = image_tools.convert_to_uint8(image_tools.resize_with_pad(wrist_img, args['resize_size'], args['resize_size']))

            # Image.fromarray(np.uint8(img)).save("./experiments/tmp/live_image.png")
            if max_angle is not None:
                text = f"step: {t}, stage: {current_stage}\nis_pre_grasp: {is_pre_grasp}\nuse_search: {use_search}\nmax_angle: {max_angle}"
            else:
                text = f"step: {t}, stage: {current_stage}\nis_pre_grasp: {is_pre_grasp}\nuse_search: {use_search}"

            tempimg = add_text_to_image(img, text)
            replay_images.append(tempimg)

            if not action_plan:
                # Finished executing previous action chunk -- compute new chunk
                # Use search-based approach only after threshold step
                if t >= args['search_start_step'] and is_pre_grasp == False:
                    use_search = True

                if use_search == True:
                    # Search for best action chunk through trajectory evaluation
                    element = {
                                "observation/image": img,
                                "observation/wrist_image": wrist_img,
                                "observation/state": np.concatenate(
                                    (obs["robot0_eef_pos"],
                                    _quat2axisangle(obs["robot0_eef_quat"]),
                                    obs["robot0_gripper_qpos"],)
                                ),
                                "prompt": str(task_description),
                                "sampling_bs": int(args['sampling_bs']),
                                "sampling_std": float(args['sampling_std']),
                                }
                    
                    # First, generate candidate action chunks
                    candidate_action_chunks = generate_candidate_action_chunks(
                        client, element, args['replan_steps'])                # candidate_action_chunks shape: (bsz, horizon, 7)
                    cost_sampled_joint_pose, cost_sampled_action, target_xyz = sample_positions(env_vox, task_description, obs, reusable_valuemap["target_objects"], reusable_valuemap["avoid_objects"]) # cost_sampled_action: list[np.ndarray]
                    print(f"target_xyz: {target_xyz}")
                    print(f"current_xyz: {obs['robot0_eef_pos']}")
                    planed_action_chunk = generate_action_sequence_fixed_steps(obs['robot0_eef_pos'], target_xyz, 20)
                    # print(f"cost_sampled_action: {cost_sampled_action}")
                    # angles = calculate_similarity_between_actions(cost_sampled_action[0], candidate_action_chunks)
                    angles = calculate_similarity_between_actions(planed_action_chunk, candidate_action_chunks)
                    logging.info(f"step: {t}, angles: {angles}")
                    
                    # # Then, select the best action chunk
                    # logging.info(f"***Searching for best action chunk using valuemap...")
                    # best_action_chunk = select_best_action_chunk_using_valuemap(
                    #     candidate_action_chunks, env, reusable_valuemap, args['replan_steps'], 
                    #     logger, t, current_episode_costs)

                    # Then, select the best action chunk using angles
                    logging.info(f"***Selecting best action chunk using angles...")
                    selected_action_chunk, max_angle = select_best_action_chunk_using_angles(
                        candidate_action_chunks, angles)
                    # print(f"selected_action_chunk: {selected_action_chunk}")
                        
                    if max_angle > 0:
                        best_action_chunk = selected_action_chunk
                    elif max_angle < 0:
                        logging.info(f"step: {t}, max_angle: {max_angle}, use cost_sampled_action")
                        best_action_chunk = planed_action_chunk

                elif use_search == False:
                    # Normal inference without search (before threshold)
                    element = {
                    "observation/image": img,
                    "observation/wrist_image": wrist_img,
                    "observation/state": np.concatenate(
                        (obs["robot0_eef_pos"],
                        _quat2axisangle(obs["robot0_eef_quat"]),
                        obs["robot0_gripper_qpos"],)
                        ),
                    "prompt": str(task_description),
                    "sampling_bs": 1,
                    "sampling_std": 1.0,
                    }
                    action_chunk = client.infer(element)["actions"]
                    assert action_chunk.shape[-2] >= args['replan_steps'], (f"We want to replan every {args['replan_steps']} steps, but policy only predicts {action_chunk.shape[-2]} steps.")
                    best_action_chunk = action_chunk[0]
                    # print(f"best_action_chunk: {best_action_chunk}")

                action_plan.extend(best_action_chunk[: args['replan_steps']])

            action = action_plan.popleft()
            # Execute action in environment
            obs, reward, done, info = env.step(action.tolist())

            # 或者机器人本体信息
            gripper_is_closed_result = is_gripper_closed(obs)
            eef_total_velocity = robot_instance._hand_total_velocity  # vx, vy, vz, rx, ry, rz 3个线速度, 3个角速度
            # 计算线速度的幅值（前3个分量）
            linear_speed = np.linalg.norm(eef_total_velocity[:3])

            linear_speed_trajectory.append(linear_speed)
            gripper_state_trajectory.append(gripper_is_closed_result)

            # 检测机械臂是否处于抓取状态
            proprioception_res = detect_pre_grasp_state(t, 
                                                        linear_speed_trajectory, 
                                                        gripper_state_trajectory,
                                                        speed_percentage_threshold = 0.3,
                                                        low_speed_window = 1,
                                                        gripper_change_lookahead = 5,
                                                        min_history_window = 20) 
                                                                                     
            is_pre_grasp = proprioception_res["is_pre_grasp"]
            is_next_stage = proprioception_res["is_stable_grasp_and_moving"]
            if is_pre_grasp == False:
                use_search = True
            else:
                use_search = False
            if done:
                task_successes += 1
                total_successes += 1
                break
            t += 1

        ##################################################################################################################
        ##################################################################################################################
        task_episodes += 1
        total_episodes += 1

        # Save cost data for current episode
        episode_data = {
            "episode_idx": episode_idx,
            "task_description": task_description,
            "success": done,
            "total_steps": t,
            "costs": current_episode_costs.copy(),
        }
        all_episode_costs.append(episode_data)

        logger.info(f"Episode {episode_idx + 1} completed - Success: {done}, Steps: {t}, Cost records: {len(current_episode_costs)}")

        suffix = "success" if done else "failure"
        task_segment = task_description.replace(" ", "_")
        video_filename = f"task{task_id:02d}_ep{episode_idx:03d}_{task_segment}_{suffix}.mp4"
        video_path = video_out_dir / video_filename
        
        imageio.mimwrite(
            video_path,
            [np.asarray(x) for x in replay_images],
            fps=18,
        )
        logging.info(f"Video saved to: {video_path}")

        # Save velocity trajectory plot
        velocity_plot_filename = f"task{task_id:02d}_ep{episode_idx:03d}_{task_segment}_{suffix}_velocity.png"
        velocity_plot_path = img_out_dir / velocity_plot_filename
        _plot_trajectory(linear_speed_trajectory, str(velocity_plot_path))

        # Save gripper state trajectory plot
        gripper_state_plot_filename = f"task{task_id:02d}_ep{episode_idx:03d}_{task_segment}_{suffix}_gripper_state.png"
        gripper_state_plot_path = img_out_dir / gripper_state_plot_filename
        _plot_trajectory(gripper_state_trajectory, str(gripper_state_plot_path))

        # Save raw trajectory data
        suffix = "success" if done else "failure"
        task_segment = task_description.replace(" ", "_")
        trajectory_filename = f"task{task_id:02d}_ep{episode_idx:03d}_{task_segment}_{suffix}_trajectory.json"
        trajectory_path = traj_raw_data_dir / trajectory_filename
        
        trajectory_data = {
            "linear_speed_trajectory": [float(x) for x in linear_speed_trajectory],
            "gripper_state_trajectory": [bool(x) for x in gripper_state_trajectory],
        }
        
        with open(trajectory_path, 'w') as f:
            json.dump(trajectory_data, f, indent=2)
        
        logging.info(f"Raw trajectory data saved to: {trajectory_path}")
        # Log current results
        logger.info(f"Episode result: {'Success' if done else 'Failure'}")
        logger.info(f"Completed episodes: {total_episodes}")
        logger.info(f"Successful episodes: {total_successes} ({total_successes / total_episodes * 100:.1f}%)")

    # Log final results
    task_success_rate = float(task_successes) / float(task_episodes) if task_episodes > 0 else 0
    total_success_rate = float(total_successes) / float(total_episodes) if total_episodes > 0 else 0

    logger.info(f"Current task success rate: {task_success_rate:.4f} ({task_success_rate * 100:.1f}%)")
    logger.info(f"Overall success rate: {total_success_rate:.4f} ({total_success_rate * 100:.1f}%)")
    logger.info(f"Current task episodes: {task_episodes}, successful: {task_successes}")
    logger.info(f"Total episodes: {total_episodes}, total successful: {total_successes}")

    # Save cost data for current task
    task_cost_file = f"./experiments/cost/{run_id}/cost_data_task_{task_id}.json"
    os.makedirs(os.path.dirname(task_cost_file), exist_ok=True)
    with open(task_cost_file, "w") as f:
        json.dump(all_episode_costs, f, indent=2, default=str)
    logger.info(f"Cost data saved to: {task_cost_file}")

    # break

# Calculate final results
final_success_rate = float(total_successes) / float(total_episodes) if total_episodes > 0 else 0

# Log final results
logger.info("=" * 60)
logger.info("Experiment completed - Final results:")
logger.info(f"Total episodes: {total_episodes}")
logger.info(f"Total successful: {total_successes}")
logger.info(f"Final success rate: {final_success_rate:.4f} ({final_success_rate * 100:.1f}%)")
logger.info("=" * 60)
logger.info(f"Experiment run ID: {run_id}")
logger.info(f"Log file: {log_filepath}")
logger.info("Experiment completed!")

print("\n✓ Evaluation complete!")

INFO:root:Videos will be saved to: experiments/videos/20251107_142800_libero_spatial_displacement, Images will be saved to: experiments/imgs/20251107_142800_libero_spatial_displacement, Trajectory raw data will be saved to: experiments/traj_raw_data/20251107_142800_libero_spatial_displacement
  0%|          | 0/10 [00:00<?, ?it/s]

[Warning]: datasets path /home/zijianwang/openpi/third_party/LIBERO-PRO/libero/libero/../datasets does not exist!
[Warning]: datasets path /home/zijianwang/openpi/third_party/LIBERO-PRO/libero/libero/../datasets does not exist!


INFO:__main__:
***Starting task: pick up the black bowl next to the ramekin and place it on the plate, This task has 2 stages.
INFO:__main__:
Task: pick up the black bowl next to the ramekin and place it on the plate
INFO:__main__:Episode 1/50
INFO:root:**All target objects**: ['akita_black_bowl_1', 'plate_1']
INFO:root:**Using target object in stage 0**: ['akita_black_bowl_1']
INFO:root:ValueMap构建完成: 目标对象: ['akita_black_bowl_1'], 避免对象: ['akita_black_bowl_2', 'cookies_1', 'glazed_rim_porcelain_ramekin_1', 'plate_1', 'wooden_cabinet_1', 'flat_stove_1']
INFO:root:ValueMap构建完成, 已恢复原始环境状态
INFO:__main__:Starting episode 1...


target_xyz: [-0.21555555  0.0242424   1.1589696 ]
current_xyz: [-2.08464661e-01  2.48894170e-12  1.17327948e+00]


INFO:root:step: 10, angles: [0.9578955020746377, 0.6461147710323315, -0.47809004414715234, 0.5380216256064744, -0.7800862397757469]
INFO:root:***Selecting best action chunk using angles...
 10%|█         | 1/10 [00:39<05:57, 39.68s/it]


KeyboardInterrupt: 